In [2]:
!pip install pandas

  Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
#!/usr/bin/env python3
"""
IXP Peering Map — Pakistani ISPs
Run this in a Jupyter notebook cell by cell.
Requires: pip install folium pandas
"""

import folium
import pandas as pd
from folium.plugins import MarkerCluster

# ── Data from PeeringDB JSON exports ─────────────────────────────────────────

ixp_data = [
    # name, city, lat, lon, isps present, capacities
    {
        "name": "DE-CIX Frankfurt",
        "city": "Frankfurt, Germany",
        "lat": 50.11, "lon": 8.68,
        "isps": {"PTCL": "300G", "Transworld": "100G", "Cybernet": "100G"}
    },
    {
        "name": "Equinix Singapore",
        "city": "Singapore",
        "lat": 1.29, "lon": 103.85,
        "isps": {"PTCL": "500G", "Transworld": "600G", "Cybernet": "100G"}
    },
    {
        "name": "AMS-IX",
        "city": "Amsterdam, Netherlands",
        "lat": 52.37, "lon": 4.89,
        "isps": {"PTCL": "200G"}
    },
    {
        "name": "LINX LON1",
        "city": "London, UK",
        "lat": 51.51, "lon": -0.13,
        "isps": {"PTCL": "10G"}
    },
    {
        "name": "DE-CIX New York",
        "city": "New York, USA",
        "lat": 40.71, "lon": -74.01,
        "isps": {"PTCL": "20G"}
    },
    {
        "name": "HKIX",
        "city": "Hong Kong",
        "lat": 22.32, "lon": 114.17,
        "isps": {"Transworld": "10G", "Cybernet": "30G"}
    },
    {
        "name": "NL-ix",
        "city": "Amsterdam, Netherlands",
        "lat": 52.38, "lon": 4.91,
        "isps": {"Transworld": "100G"}
    },
    {
        "name": "SH-IX",
        "city": "Rome, Italy",
        "lat": 41.90, "lon": 12.50,
        "isps": {"PTCL": "20G", "Transworld": "100G", "Cybernet": "10G"}
    },
    {
        "name": "UAE-IX",
        "city": "Dubai, UAE",
        "lat": 25.20, "lon": 55.27,
        "isps": {"PTCL": "10G", "Cybernet": "10G"}
    },
    {
        "name": "Equinix Muscat",
        "city": "Muscat, Oman",
        "lat": 23.59, "lon": 58.59,
        "isps": {"Transworld": "10G"}
    },
    {
        "name": "Oman-IX",
        "city": "Muscat, Oman",
        "lat": 23.62, "lon": 58.60,
        "isps": {"Transworld": "10G"}
    },
    {
        "name": "DE-CIX Marseille",
        "city": "Marseille, France",
        "lat": 43.30, "lon": 5.37,
        "isps": {"Cybernet": "20G"}
    },
    {
        "name": "NetIX",
        "city": "Sofia, Bulgaria",
        "lat": 42.70, "lon": 23.32,
        "isps": {"Cybernet": "50G"}
    },
    {
        "name": "PIE Karachi (DE-CIX)",
        "city": "Karachi, Pakistan",
        "lat": 24.86, "lon": 67.01,
        "isps": {"PTCL": "100G"},
        "is_pakistan": True
    },
]

# ISPs with NO IXP presence
no_ixp_isps = ["Nayatel", "Nova", "Zcom"]

# ISP colors
isp_colors = {
    "PTCL":       "#2a78d6",
    "Transworld":  "#1baf7a",
    "Cybernet":    "#eda100",
}

# ── Build map ─────────────────────────────────────────────────────────────────

m = folium.Map(
    location=[30, 40],
    zoom_start=3,
    tiles="CartoDB positron",
)

for ixp in ixp_data:
    isps = ixp["isps"]
    is_pk = ixp.get("is_pakistan", False)

    # Build popup HTML
    isp_lines = "".join(
        f"<span style='color:{isp_colors.get(isp, '#888')}'>"
        f"<b>{isp}</b>: {cap}</span><br>"
        for isp, cap in isps.items()
    )
    popup_html = f"""
    <div style='font-family:sans-serif;font-size:13px;min-width:160px'>
        <b>{ixp['name']}</b><br>
        <span style='color:#888'>{ixp['city']}</span><br><br>
        {isp_lines}
    </div>
    """

    # Color: if Pakistan IXP use red, else color by first ISP
    if is_pk:
        color = "red"
        icon = folium.Icon(color="red", icon="star", prefix="fa")
    elif len(isps) == 1:
        isp = list(isps.keys())[0]
        color = isp_colors.get(isp, "#888")
        icon = None
    else:
        color = "purple"  # multiple ISPs
        icon = None

    # Radius scales with number of ISPs
    radius = 8 + len(isps) * 4

    folium.CircleMarker(
        location=[ixp["lat"], ixp["lon"]],
        radius=radius,
        color="white",
        weight=1.5,
        fill=True,
        fill_color="red" if is_pk else (
            isp_colors[list(isps.keys())[0]] if len(isps) == 1 else "#7c5cbf"
        ),
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=ixp["name"],
    ).add_to(m)

# Pakistan home marker
folium.Marker(
    location=[30.37, 69.34],
    popup="<b>Pakistan</b><br>Home country",
    tooltip="Pakistan",
    icon=folium.Icon(color="darkred", icon="home", prefix="fa"),
).add_to(m)

# ── Legend ────────────────────────────────────────────────────────────────────

legend_html = """
<div style="
    position: fixed;
    bottom: 30px; left: 30px;
    background: white;
    border: 1px solid #ccc;
    border-radius: 8px;
    padding: 12px 16px;
    font-family: sans-serif;
    font-size: 13px;
    z-index: 1000;
    line-height: 2;
">
    <b>Pakistani ISPs at IXPs</b><br>
    <span style="color:#2a78d6">&#9679;</span> PTCL (AS17557)<br>
    <span style="color:#1baf7a">&#9679;</span> Transworld (AS38193)<br>
    <span style="color:#eda100">&#9679;</span> Cybernet (AS9541)<br>
    <span style="color:#7c5cbf">&#9679;</span> Multiple ISPs<br>
    <span style="color:red">&#9733;</span> PIE Karachi (only PK IXP)<br>
    <hr style="margin:6px 0">
    <span style="color:#aaa">&#9679;</span> Nayatel — no IXP presence<br>
    <span style="color:#aaa">&#9679;</span> Nova — no IXP presence<br>
    <span style="color:#aaa">&#9679;</span> Zcom — no IXP presence<br>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# ── Save and display ──────────────────────────────────────────────────────────

m.save("ixp_map.html")
print("Saved to ixp_map.html")

# In Jupyter notebook, display inline:
m

Saved to ixp_map.html
